In [ ]:
#
# Copyright (C) 2018 Pico Technology Ltd. See LICENSE file for terms.
#
# PS3000A BLOCK MODE EXAMPLE
# This example opens a 3000a driver device, sets up one channels and a trigger then collects a block of data.
# This data is then plotted as mV against time in ns.

import ctypes
from picosdk.ps3000a import ps3000a as ps
import numpy as np
import matplotlib.pyplot as plt
from picosdk.functions import adc2mV, assert_pico_ok

In [ ]:
# Create chandle and status ready for use
status = {}
chandle = ctypes.c_int16()

# Opens the device/s
status["openunit"] = ps.ps3000aOpenUnit(ctypes.byref(chandle), None)

try:
    assert_pico_ok(status["openunit"])
except:

    # powerstate becomes the status number of openunit
    powerstate = status["openunit"]

    # If powerstate is the same as 282 then it will run this if statement
    if powerstate == 282:
        # Changes the power input to "PICO_POWER_SUPPLY_NOT_CONNECTED"
        print("PICO_POWER_SUPPLY_NOT_CONNECTED")
        status["ChangePowerSource"] = ps.ps3000aChangePowerSource(chandle, 282)
        # If the powerstate is the same as 286 then it will run this if statement
    elif powerstate == 286:
        # Changes the power input to "PICO_USB3_0_DEVICE_NON_USB3_0_PORT"
        print("PICO_USB3_0_DEVICE_NON_USB3_0_PORT")
        status["ChangePowerSource"] = ps.ps3000aChangePowerSource(chandle, 286)
    else:
        raise RuntimeError("Unable to open Picoscope device.")

    assert_pico_ok(status["ChangePowerSource"])

In [ ]:
# Set up channel A
# handle = chandle
# channel = ps.PS3000A_CHANNEL['PS3000A_CHANNEL_A'] = 0
# enabled = 1
# coupling type = ps.PS3000A_COUPLING['PS3000A_DC'] = 1
# range = PS3000A_5V = 8
# 0  == PS3000A_10MV:  ±10 mV - нет
# 1  == PS3000A_20MV:  ±20 mV - нет
# 2  == PS3000A_50MV:  ±50 mV
# 3  == PS3000A_100MV: ±100 mV
# 4  == PS3000A_200MV: ±200 mV
# 5  == PS3000A_500MV: ±500 mV
# 6  == PS3000A_1V:    ±1 V
# 7  == PS3000A_2V:    ±2 V
# 8  == PS3000A_5V:    ±5 V
# 9  == PS3000A_10V:   ±10 V
# 10 == PS3000A_20V:   ±20 V
# analogue offset = 0 V
chARange = 8
status["setChA"] = ps.ps3000aSetChannel(chandle, 0, 1, 1, chARange, 0)
assert_pico_ok(status["setChA"])

In [ ]:
# Set up channel B
# handle = chandle
# channel = ps.PS3000A_CHANNEL['PS3000A_CHANNEL_B'] = 1
# enabled = 1
# coupling type = ps.PS3000A_COUPLING['PS3000A_DC'] = 1
# range = PS3000A_5V = 7
# 0  == PS3000A_10MV:  ±10 mV - нет
# 1  == PS3000A_20MV:  ±20 mV - нет
# 2  == PS3000A_50MV:  ±50 mV
# 3  == PS3000A_100MV: ±100 mV
# 4  == PS3000A_200MV: ±200 mV
# 5  == PS3000A_500MV: ±500 mV
# 6  == PS3000A_1V:    ±1 V
# 7  == PS3000A_2V:    ±2 V
# 8  == PS3000A_5V:    ±5 V
# 9  == PS3000A_10V:   ±10 V
# 10 == PS3000A_20V:   ±20 V
# analogue offset = 0 V
chBRange = 7
status["setChB"] = ps.ps3000aSetChannel(chandle, 0, 1, 1, chBRange, 0)
assert_pico_ok(status["setChB"])

In [ ]:
# Sets up single trigger
# Handle = Chandle
# Source = ps3000A_channel_B = 0
# Enable = 0
# Threshold = 1024 ADC counts
# Direction = ps3000A_Falling = 3
# Delay = 0
# autoTrigger_ms = 1000
status["trigger"] = ps.ps3000aSetSimpleTrigger(chandle, 1, 0, 1024, 3, 0, 1000)
assert_pico_ok(status["trigger"])

In [ ]:
# Setting the number of sample to be collected
preTriggerSamples = 0
postTriggerSamples = 20000
maxsamples = preTriggerSamples + postTriggerSamples

# Gets timebase infomation
# WARNING: When using this example it may not be possible to access all Timebases as all channels are enabled by default when opening the scope.  
# To access these Timebases, set any unused analogue channels to off.
# Handle = chandle
# Timebase = 2 = timebase
# Nosample = maxsamples
# TimeIntervalNanoseconds = ctypes.byref(timeIntervalns)
# MaxSamples = ctypes.byref(returnedMaxSamples)
# Segement index = 0
timebase = 1252 # 10 mks
timeIntervalns = ctypes.c_float()
returnedMaxSamples = ctypes.c_int16()
status["GetTimebase"] = ps.ps3000aGetTimebase2(chandle, timebase, maxsamples, ctypes.byref(timeIntervalns), 1, ctypes.byref(returnedMaxSamples), 0)
assert_pico_ok(status["GetTimebase"])

In [ ]:
# Creates a overlow location for data
overflow = ctypes.c_int16()
# Creates converted types maxsamples
cmaxSamples = ctypes.c_int32(maxsamples)

# Starts the block capture
# Handle = chandle
# Number of prTriggerSamples
# Number of postTriggerSamples
# Timebase = 2 = 4ns (see Programmer's guide for more information on timebases)
# time indisposed ms = None (This is not needed within the example)
# Segment index = 0
# LpRead = None
# pParameter = None
status["runblock"] = ps.ps3000aRunBlock(chandle, preTriggerSamples, postTriggerSamples, timebase, 1, None, 0, None, None)
assert_pico_ok(status["runblock"])

# Create buffers ready for assigning pointers for data collection
bufferAMax = (ctypes.c_int16 * maxsamples)()
# bufferAMin = (ctypes.c_int16 * maxsamples)() # used for downsampling
bufferBMax = (ctypes.c_int16 * maxsamples)()
# bufferBMin = (ctypes.c_int16 * maxsamples)() # used for downsampling

# Setting the data buffer location for data collection from channel A
# Handle = Chandle
# source = ps3000A_channel_A = 0
# Buffer max = ctypes.byref(bufferAMax)
# Buffer min = ctypes.byref(bufferAMin)
# Buffer length = maxsamples
# Segment index = 0
# Ratio mode = ps3000A_Ratio_Mode_None = 0
status["SetDataBuffersA"] = ps.ps3000aSetDataBuffers(chandle,
                                                    ps.PS3000A_CHANNEL['PS3000A_CHANNEL_A'],    # source = PS3000A_CHANNEL_A = 0
                                                    ctypes.byref(bufferAMax),
                                                    None,
                                                    maxsamples, # buffer length = maxSamples
                                                    0,          # segment index = 0
                                                    ps.PS3000A_RATIO_MODE['PS3000A_RATIO_MODE_NONE'] # ratio mode = PS3000A_RATIO_MODE_NONE = 0
assert_pico_ok(status["SetDataBuffersA"])
status["SetDataBuffersB"] = ps.ps3000aSetDataBuffers(chandle,
                                                    ps.PS3000A_CHANNEL['PS3000A_CHANNEL_B'],    # source = PS3000A_CHANNEL_B = 1
                                                    ctypes.byref(bufferBMax),
                                                    None,
                                                    maxsamples, # buffer length = maxSamples
                                                    0,          # segment index = 0
                                                    ps.PS3000A_RATIO_MODE['PS3000A_RATIO_MODE_NONE'] # ratio mode = PS3000A_RATIO_MODE_NONE = 0
assert_pico_ok(status["SetDataBuffersB"])

# Creates a overlow location for data
overflow = (ctypes.c_int16 * 10)()
# Creates converted types maxsamples
cmaxSamples = ctypes.c_int32(maxsamples)

# Checks data collection to finish the capture
ready = ctypes.c_int16(0)
check = ctypes.c_int16(0)
while ready.value == check.value:
    status["isReady"] = ps.ps3000aIsReady(chandle, ctypes.byref(ready))

# Handle = chandle
# start index = 0
# noOfSamples = ctypes.byref(cmaxSamples)
# DownSampleRatio = 0
# DownSampleRatioMode = 0
# SegmentIndex = 0
# Overflow = ctypes.byref(overflow)

status["GetValues"] = ps.ps3000aGetValues(chandle, 0, ctypes.byref(cmaxSamples), 0, 0, 0, ctypes.byref(overflow))
assert_pico_ok(status["GetValues"])

# Finds the max ADC count
# Handle = chandle
# Value = ctype.byref(maxADC)
maxADC = ctypes.c_int16()
status["maximumValue"] = ps.ps3000aMaximumValue(chandle, ctypes.byref(maxADC))
assert_pico_ok(status["maximumValue"])

# Converts ADC from channels to mV
adc2mVChAMax = adc2mV(bufferAMax, chARange, maxADC)
adc2mVChBMax = adc2mV(bufferBMax, chBRange, maxADC)

# Creates the time data
time = np.linspace(0, (cmaxSamples.value - 1) * timeIntervalns.value, cmaxSamples.value)


In [ ]:
# Plots the data from channel A onto a graph
plt.plot(time, adc2mVChAMax[:])
plt.plot(time, adc2mVChBMax[:])
plt.xlabel('Time (ns)')
plt.ylabel('Voltage (mV)')
plt.show()

In [ ]:

# Stops the scope
# Handle = chandle
status["stop"] = ps.ps3000aStop(chandle)
assert_pico_ok(status["stop"])

# Closes the unit
# Handle = chandle
status["close"] = ps.ps3000aCloseUnit(chandle)
assert_pico_ok(status["close"])

# Displays the staus returns
print(status)